<a href="https://raw.githubusercontent.com/tsg-/oneccl-tutorial/master/notebooks/03b_allgather.ipynb" download>⬇️ Download this notebook</a>

# Allgather: Sequence Parallelism

> **Concepts first?** [Foundations §3.2](../chapters/00_foundations) explains what
> allgather does and why ring is the right algorithm. [When to Use Which Collective](../chapters/03_when_to_use)
> covers the SP decision tree.

## What allgather does

Each rank contributes a distinct chunk of data. After allgather, every rank holds all
chunks concatenated in rank order.

```
Before:
  Rank 0: [tokens  0..127]     (my sequence shard)
  Rank 1: [tokens 128..255]
  Rank 2: [tokens 256..383]
  Rank 3: [tokens 384..511]

After allgather:
  Rank 0: [tokens 0..511]      ← full sequence
  Rank 1: [tokens 0..511]
  Rank 2: [tokens 0..511]
  Rank 3: [tokens 0..511]
```

## Inference use case: Sequence Parallelism (SP)

When running with long context (e.g., 128K tokens), the KV cache and attention computation
are split across ranks along the sequence dimension:

- Each GPU holds `seq_len / N` tokens worth of KV cache
- Before computing attention, each GPU needs the full key/value context
- **Allgather reconstructs the full KV** so every GPU can attend over all tokens
- After attention, each GPU computes on its own query shard

The allgather is the communication cost of sequence parallelism. Its size scales with
sequence length and is larger than typical TP allreduces.

## Why Ring algorithm for allgather

Allgather is the second phase of Ring Allreduce (see [Foundations §5.2](../chapters/00_foundations)).
The Ring Allgather runs N-1 rotation steps where each rank forwards one chunk per step.
Every link carries msg/N bytes per step — bandwidth-optimal, and topology-aware on NUMA.

:::{note}
All cells use the **launcher pattern**: the MPI workload is written to a temp script and
executed via `mpirun` as a subprocess. Run cells from a single-rank Jupyter kernel.
:::

## 0. Environment Check

In [ ]:
import subprocess, os

def check(label, cmd, expect_in=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0 and (expect_in is None or expect_in in result.stdout + result.stderr)
    print(f"{'OK' if ok else 'FAIL'}  {label}")
    if not ok:
        print(f"     {result.stdout.strip()[:200] or result.stderr.strip()[:200]}")
    return ok

check("mpirun available",          "which mpirun")
check("Intel MPI loaded",          "mpirun --version", expect_in="Intel")
check("PyTorch installed",         "python -c 'import torch; print(torch.__version__)'")
check("oneCCL bindings installed", "python -c 'import oneccl_bindings_for_pytorch'")
check("XPU available",             "python -c 'import torch; assert torch.xpu.is_available()'")

## 1. Basic Allgather — Correctness Verification

Each rank contributes a tensor filled with its rank ID. After allgather, every rank should
hold `[0, 0, ..., 1, 1, ..., 2, 2, ..., 3, 3, ...]` — the chunks from each rank in order.

In [ ]:
%%writefile /tmp/ccl_allgather_basic.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ.setdefault("CCL_ATL_TRANSPORT", "ofi")
os.environ.setdefault("CCL_WORKER_COUNT", "1")
os.environ.setdefault("CCL_LOG_LEVEL", "warn")

dist.init_process_group(backend="ccl")

rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Simulate: each rank holds a shard of the sequence KV cache
# Shape: (seq_shard, hidden) where seq_shard = total_seq / world_size
total_seq = 512
hidden    = 1024
seq_shard = total_seq // world_size

# Each rank fills its shard with its rank value (so we can verify ordering)
my_shard = torch.full((seq_shard, hidden), fill_value=float(rank),
                       dtype=torch.bfloat16, device=device)

# Output buffer: full sequence, all ranks
full_seq = torch.zeros((total_seq, hidden), dtype=torch.bfloat16, device=device)

if rank == 0:
    print(f"Shard size per rank: {my_shard.shape}")
    print(f"Full output size:    {full_seq.shape}")
    print(f"My shard[0, 0] = {my_shard[0, 0].item():.0f} (should be {rank})")

# Allgather: distribute list of output tensors, one per rank
# torch.distributed.all_gather expects a list of same-shape tensors
output_list = list(torch.chunk(full_seq, world_size, dim=0))
dist.all_gather(output_list, my_shard)
torch.xpu.synchronize(device)

# Verify: chunk i should be filled with value i
errors = 0
for i in range(world_size):
    expected = float(i)
    actual   = output_list[i][0, 0].item()
    if abs(actual - expected) > 0.01:
        errors += 1
        print(f"[rank {rank}] chunk {i}: expected {expected}, got {actual}")

status = "PASS" if errors == 0 else f"FAIL ({errors} errors)"
print(f"[rank {rank}] allgather correctness: {status}")
print(f"[rank {rank}] full_seq[0,0]={output_list[0][0,0].item():.0f} "
      f"full_seq[seq_shard,0]={output_list[1][0,0].item():.0f}  "
      f"(expected 0, 1)")

dist.destroy_process_group()

In [ ]:
import subprocess
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_allgather_basic.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 2. Allgather with Non-Uniform Shards (allgatherv)

In practice, sequence shards may not divide evenly (e.g., seq_len=513 across 4 ranks).
`allgatherv` (the variable-length variant) handles this.

In PyTorch, use `dist.all_gather_into_tensor` with explicit output sizing, or pass
`output_split_sizes` to `dist.all_to_all_single` for the alltoallv pattern.

For the simpler case where the shard sizes are known and fixed at runtime:

In [ ]:
%%writefile /tmp/ccl_allgather_uneven.py
import os
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ.setdefault("CCL_ATL_TRANSPORT", "ofi")
os.environ.setdefault("CCL_WORKER_COUNT", "1")
os.environ.setdefault("CCL_LOG_LEVEL", "warn")

dist.init_process_group(backend="ccl")

rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Uneven sequence: 513 tokens across 4 ranks
# Ranks 0 gets 129, ranks 1-3 get 128
total_seq   = 513
hidden      = 256
shard_sizes = [total_seq // world_size + (1 if i < total_seq % world_size else 0)
               for i in range(world_size)]
my_size     = shard_sizes[rank]

my_shard = torch.full((my_size, hidden), fill_value=float(rank),
                       dtype=torch.bfloat16, device=device)

# allgatherv: gather into pre-allocated tensors of correct size
output_chunks = [
    torch.empty((shard_sizes[i], hidden), dtype=torch.bfloat16, device=device)
    for i in range(world_size)
]
dist.all_gather(output_chunks, my_shard)
torch.xpu.synchronize(device)

if rank == 0:
    print(f"Shard sizes: {shard_sizes}  (total={sum(shard_sizes)})")
    for i, chunk in enumerate(output_chunks):
        print(f"  chunk[{i}]: shape={tuple(chunk.shape)}  value={chunk[0,0].item():.0f}  "
              f"({'PASS' if abs(chunk[0,0].item() - i) < 0.01 else 'FAIL'})")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_allgather_uneven.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 3. Latency Benchmark — Allgather vs. Message Size

Sequence parallelism allgather sizes are much larger than TP allreduce sizes.
A 32K token context with hidden=8192 in BF16 per rank: 32768/4 × 8192 × 2 = 128 MB per shard.
The allgather collects 4 shards = 512 MB total output — bandwidth-bound by definition.

This benchmark sweeps from small (< 1 MB, latency-bound) to large (> 100 MB, bandwidth-bound).

In [ ]:
%%writefile /tmp/ccl_allgather_bench.py
import os, time, json
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

WARMUP = 5
ITERS  = 30

# Per-rank shard sizes (allgather output is world_size times larger)
shard_bytes_list = [
    256  * 1024,      # 256 KB shard  → 1 MB output
    1    * 1024**2,   # 1 MB shard    → 4 MB output
    16   * 1024**2,   # 16 MB shard   → 64 MB output
    128  * 1024**2,   # 128 MB shard  → 512 MB output (32K ctx, hidden=8192, TP=4)
]

results = []
for shard_bytes in shard_bytes_list:
    nelems      = shard_bytes // 2  # BF16
    my_shard    = torch.ones(nelems, dtype=torch.bfloat16, device=device)
    output_list = [torch.empty(nelems, dtype=torch.bfloat16, device=device)
                   for _ in range(world_size)]

    for _ in range(WARMUP):
        dist.all_gather(output_list, my_shard)
        torch.xpu.synchronize(device)
    dist.barrier()

    times = []
    for _ in range(ITERS):
        t0 = time.perf_counter()
        dist.all_gather(output_list, my_shard)
        torch.xpu.synchronize(device)
        times.append((time.perf_counter() - t0) * 1e3)  # ms
    dist.barrier()

    times.sort()
    if rank == 0:
        total_bytes = shard_bytes * world_size
        p50 = times[len(times)//2]
        p95 = times[int(len(times)*0.95)]
        # Bus BW: allgather traffic = (N-1)/N * total_output_size
        bus_bw = (world_size-1)/world_size * total_bytes / (p50 * 1e-3) / 1e9
        results.append({
            "shard_mb": shard_bytes // 1024**2,
            "output_mb": total_bytes // 1024**2,
            "p50_ms": round(p50, 2),
            "p95_ms": round(p95, 2),
            "bus_gbps": round(bus_bw, 1)
        })

if rank == 0:
    print(f"{'Shard':>8} {'Output':>8} {'p50 (ms)':>10} {'p95 (ms)':>10} {'BW (GB/s)':>11}")
    print("-" * 52)
    for r in results:
        print(f"{r['shard_mb']:>6}MB  {r['output_mb']:>6}MB  "
              f"{r['p50_ms']:>10.2f}  {r['p95_ms']:>10.2f}  {r['bus_gbps']:>9.1f}")
    print()
    print(json.dumps(results))

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_allgather_bench.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## 4. SP Attention Pattern: Allgather → Compute → (no Allreduce needed)

In Sequence Parallelism, the pattern around an attention block is:

```
1. Each rank holds q[shard], k[shard], v[shard]
2. Allgather k and v across all ranks → full k, v on every rank
3. Each rank computes attention(q[shard], k_full, v_full)
   → output is already sharded (rank only computed for its q shard)
4. No allreduce needed! Each rank's output is its own query result.
```

This is cheaper than TP allreduce because you do 2 allgathers (k, v) and no allreduce,
vs. TP which does 1 allreduce per attention output.

The trade-off: each rank must store the full k, v tensors — higher memory pressure.
For long context, k and v can be very large, making SP memory-intensive.

In [ ]:
%%writefile /tmp/ccl_sp_pattern.py
import os, time
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_LOG_LEVEL"]     = "error"

dist.init_process_group(backend="ccl")
rank       = dist.get_rank()
world_size = dist.get_world_size()
device     = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Model config: Llama-3-8B, heads=32, head_dim=128, hidden=4096
total_seq = 8192
num_heads = 8          # GQA: 8 KV heads
head_dim  = 128
kv_dim    = num_heads * head_dim  # 1024
seq_shard = total_seq // world_size

# Each rank holds its k and v shards
k_shard = torch.randn(seq_shard, kv_dim, dtype=torch.bfloat16, device=device)
v_shard = torch.randn(seq_shard, kv_dim, dtype=torch.bfloat16, device=device)

# Full k, v buffers (all ranks need these for attention)
k_full_chunks = [torch.empty(seq_shard, kv_dim, dtype=torch.bfloat16, device=device)
                 for _ in range(world_size)]
v_full_chunks = [torch.empty(seq_shard, kv_dim, dtype=torch.bfloat16, device=device)
                 for _ in range(world_size)]

WARMUP = 5
ITERS  = 20

for _ in range(WARMUP):
    dist.all_gather(k_full_chunks, k_shard)
    dist.all_gather(v_full_chunks, v_shard)
    torch.xpu.synchronize(device)
dist.barrier()

times = []
for _ in range(ITERS):
    t0 = time.perf_counter()
    # Gather K and V (can be overlapped in practice)
    work_k = dist.all_gather(k_full_chunks, k_shard, async_op=True)
    work_v = dist.all_gather(v_full_chunks, v_shard, async_op=True)
    work_k.wait()
    work_v.wait()
    torch.xpu.synchronize(device)
    times.append((time.perf_counter() - t0) * 1e3)
dist.barrier()

if rank == 0:
    times.sort()
    p50 = times[len(times)//2]
    kv_bytes = 2 * seq_shard * kv_dim * 2  # 2 (K+V) * elements * BF16
    total_output = kv_bytes * world_size
    # Bus BW for ring allgather: (N-1)/N * total_output / time
    bus_bw = (world_size-1)/world_size * total_output / (p50*1e-3) / 1e9
    print(f"SP KV allgather pattern:")
    print(f"  seq_shard: {seq_shard} tokens/rank, kv_dim: {kv_dim}")
    print(f"  per-rank K shard: {seq_shard * kv_dim * 2 / 1024:.1f} KB")
    print(f"  p50 (K+V allgather): {p50:.2f} ms")
    print(f"  effective bus bandwidth: {bus_bw:.1f} GB/s")

dist.destroy_process_group()

In [ ]:
result = subprocess.run(
    "mpirun -n 4 -ppn 4 python /tmp/ccl_sp_pattern.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## Summary

| Takeaway | Detail |
|---|---|
| **Allgather = gather all chunks onto all ranks** | Output size = input size × world_size |
| **SP uses 2 allgathers per attention layer** | K and V; no allreduce needed |
| **Sizes are larger than TP allreduce** | Seq-parallel KV at 32K ctx can be 100s of MB |
| **Ring is correct on NUMA** | Same reasoning as allreduce — bandwidth-optimal, topology-aware |
| **Async allgather for K and V** | Fire both, wait both — minimal latency increase vs. serial |
| **Memory tradeoff** | SP requires full K/V on each rank; TP does not — choose based on memory budget |

**Next:** [Alltoall — MoE Expert Routing](03c_alltoall_moe.ipynb)